# Reliable and Survey-Aware ML for Childhood Stunting Screening under Anthropometric Measurement Uncertainty
Cross-check notebook (Google Colab). Run cells top to bottom.

**Before running:** upload to Google Drive folder `MyDrive/stunting_framework/data/` the files
`SKI 2023 Balita 0-59 Bulan_asli labels.xlsx` and `lenanthro.txt`.

Expected runtime on Colab CPU: ~40-60 min (Random Forest CV is the slowest step). Numbers should match `results/*.csv` in the project folder (seed = 42); tiny differences (<0.002 AUROC) can occur with other scikit-learn versions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJ = '/content/drive/MyDrive/stunting_framework'
os.makedirs(PROJ + '/scripts', exist_ok=True); os.makedirs(PROJ + '/data', exist_ok=True)
os.environ['SKI_PROJECT_DIR'] = PROJ
!pip -q install scikit-learn==1.8.0 openpyxl joblib
import sklearn, pandas, numpy; print(sklearn.__version__, pandas.__version__, numpy.__version__)

## Plot style helper (`plot_style.py`)

In [ ]:
%%writefile {PROJ}/scripts/plot_style.py
"""
Gaya visualisasi konsisten untuk semua figure eksperimen dalam proyek ini.
Panggil apply_style() sekali di awal skrip, sebelum membuat plot apa pun.

Kenapa ini penting: tanpa gaya konsisten, tiap eksperimen menghasilkan
figure dengan font/warna/ukuran berbeda-beda -- menyulitkan pembaca
membandingkan lintas eksperimen dan menyulitkan proses "tempel ke naskah
jurnal" karena harus dirapikan ulang satu-satu. Palet warna di bawah
dipilih agar tetap bisa dibedakan oleh pembaca buta warna (deuteranopia/
protanopia-safe, diadaptasi dari palet kualitatif Okabe-Ito).
"""

import matplotlib.pyplot as plt

# Palet kualitatif aman-buta-warna (Okabe & Ito, 2008) -- urutan dipakai
# berulang untuk kategori ke-1, ke-2, dst.
COLORBLIND_SAFE_PALETTE = [
    "#0072B2",  # biru
    "#D55E00",  # oranye kemerahan
    "#009E73",  # hijau
    "#CC79A7",  # magenta
    "#E69F00",  # kuning-oranye
    "#56B4E9",  # biru muda
    "#F0E442",  # kuning
    "#000000",  # hitam (aksen/baseline)
]


def apply_style(base_font_size: int = 11) -> None:
    """Terapkan rcParams matplotlib yang konsisten untuk seluruh proyek."""
    plt.rcParams.update({
        "figure.dpi": 150,          # tampilan di layar
        "savefig.dpi": 300,          # cukup tajam untuk cetak/jurnal
        "font.size": base_font_size,
        "axes.titlesize": base_font_size + 2,
        "axes.labelsize": base_font_size,
        "axes.titleweight": "bold",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
        "legend.frameon": False,
        "figure.autolayout": True,
        "axes.prop_cycle": plt.cycler(color=COLORBLIND_SAFE_PALETTE),
        "savefig.bbox": "tight",
    })


def annotate_source(fig, text: str) -> None:
    """Tambahkan caption kecil (sumber data, catatan cara baca grafik, dll.)
    di pojok kiri bawah figure -- praktik baik untuk figure yang akan dipakai
    di bab hasil/naskah jurnal, supaya pembaca (dan diri sendiri 6 bulan lagi)
    tahu data ini dari mana atau bagaimana cara membacanya. Gunakan fungsi ini
    (bukan menulis fig.text(...) manual) supaya posisi & gaya caption konsisten
    di semua figure proyek ini -- lihat pemakaiannya di comparison_table.py."""
    fallback_size = max(plt.rcParams.get("font.size", 11) - 3, 7)
    fig.text(0.01, -0.03, text, fontsize=fallback_size, color="#555555", ha="left", va="top")


## Shared configuration & metric functions (`common.py`)

In [ ]:
%%writefile {PROJ}/scripts/common.py
"""
common.py - shared configuration and helper functions
Reliable and Survey-Aware ML for Childhood Stunting Risk Screening
Under Anthropometric Measurement Uncertainty (SKI 2023)

All scripts import this module. Change DATA_DIR / OUT_DIR for Google Colab
(see 00_colab_setup.md or the notebook's first cell).
"""
import os
import json
import numpy as np
import pandas as pd
from scipy.stats import norm

# ------------------------------------------------------------------ paths
BASE_DIR = os.environ.get(
    "SKI_PROJECT_DIR",
    os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
DATA_DIR = os.path.join(BASE_DIR, "data")
RES_DIR = os.path.join(BASE_DIR, "results")
FIG_DIR = os.path.join(BASE_DIR, "figures")
for _d in (RES_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)

RAW_XLSX = os.path.join(DATA_DIR, "SKI 2023 Balita 0-59 Bulan_asli labels.xlsx")
RAW_CACHE = os.path.join(DATA_DIR, "ski_raw.pkl")
LMS_FILE = os.path.join(DATA_DIR, "lenanthro.txt")
ANALYTIC = os.path.join(DATA_DIR, "analytic.pkl")

# ------------------------------------------------------------------ config
SEED = 42
N_FOLDS = 5
TEST_FRAC = 0.20                                   # share of PSUs in locked test set
SPATIAL_HOLDOUT = ["Sumatera Barat", "Bali", "Maluku"]
STUNT_CUT = -2.0
# measurement-error scenarios (random error SD in cm); see manuscript Sec. 2.6
SIGMA_CM_SCENARIOS = [0.5, 1.0, 1.5]
SIGMA_CM_MAIN = 1.0
ALPHA_MAIN = 0.10          # tolerated probability that a confirmed label is wrong
ALPHA_GRID = [0.05, 0.10, 0.20]
POSITION_BIAS_CM = 0.7     # WHO recumbent-vs-standing difference

W = "w"            # survey weight column in analytic file
PSU = "psu"
STRATA = "strata"


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=float)


# ------------------------------------------------------------------ LMS
def load_lms():
    lms = pd.read_csv(LMS_FILE, sep="\t")
    lms.columns = [c.strip().lower() for c in lms.columns]
    return lms  # sex(1=M,2=F), age(days 0..1856), l, m, s, loh


def lms_lookup(lms, sex, age_days):
    key = pd.DataFrame({"sex": sex.astype(int), "age": age_days.astype(int)})
    out = key.merge(lms[["sex", "age", "l", "m", "s"]], on=["sex", "age"], how="left")
    return out["l"].values, out["m"].values, out["s"].values


def haz_from_height(h, L, M, S):
    """WHO LMS z-score. For length/height L==1 so z=(h/M-1)/S."""
    return (np.power(h / M, L) - 1.0) / (L * S)


def z_per_cm(M, S, L=1.0):
    """dz/dh at the observed point (L=1 -> 1/(M*S)); SD units per 1 cm."""
    return 1.0 / (M * S)


# ------------------------------------------------------------------ weighted metrics
def w_auc(y, p, w):
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(y, p, sample_weight=w)


def w_auprc(y, p, w):
    from sklearn.metrics import average_precision_score
    return average_precision_score(y, p, sample_weight=w)


def w_brier(y, p, w):
    return float(np.sum(w * (p - y) ** 2) / np.sum(w))


def w_ece(y, p, w, n_bins=10):
    bins = np.clip((p * n_bins).astype(int), 0, n_bins - 1)
    ece, W_ = 0.0, np.sum(w)
    for b in range(n_bins):
        m = bins == b
        if m.sum() == 0:
            continue
        wb = w[m]
        ece += wb.sum() / W_ * abs(np.average(y[m], weights=wb) - np.average(p[m], weights=wb))
    return float(ece)


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def _wlogreg(X, y, w, offset=None, iters=50):
    """Weighted logistic regression via IRLS (unpenalised). Returns coef."""
    n, k = X.shape
    beta = np.zeros(k)
    off = np.zeros(n) if offset is None else offset
    for _ in range(iters):
        eta = X @ beta + off
        mu = 1 / (1 + np.exp(-eta))
        Wd = w * mu * (1 - mu) + 1e-12
        z = eta - off + (y - mu) / (mu * (1 - mu) + 1e-12)
        XtW = X.T * Wd
        new = np.linalg.solve(XtW @ X, XtW @ z)
        if np.max(np.abs(new - beta)) < 1e-8:
            beta = new
            break
        beta = new
    return beta


def cal_slope_intercept(y, p, w):
    """Calibration slope (logit recalibration) and calibration-in-the-large
    intercept (slope fixed at 1 via offset). Ideal: slope 1, intercept 0."""
    lp = _logit(p)
    X = np.column_stack([np.ones_like(lp), lp])
    slope = _wlogreg(X, y.astype(float), w)[1]
    intercept = _wlogreg(np.ones((len(lp), 1)), y.astype(float), w, offset=lp)[0]
    return float(slope), float(intercept)


def all_metrics(y, p, w):
    y = np.asarray(y).astype(int); p = np.asarray(p, float); w = np.asarray(w, float)
    try:
        s, i = cal_slope_intercept(y, p, w)
    except np.linalg.LinAlgError:      # constant predictions (prevalence baseline)
        s, i = np.nan, float(np.log(np.average(y, weights=w) / (1 - np.average(y, weights=w)))
                             - np.log(p.mean() / (1 - p.mean())))
    return dict(n=int(len(y)), prev_w=float(np.average(y, weights=w)),
                auroc=w_auc(y, p, w), auprc=w_auprc(y, p, w),
                brier=w_brier(y, p, w), ece=w_ece(y, p, w),
                cal_slope=s, cal_intercept=i)


def psu_bootstrap(y, p, w, psu, fn, B=200, seed=SEED):
    """Cluster (PSU) bootstrap percentile CI for metric fn(y,p,w)."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y); p = np.asarray(p); w = np.asarray(w); psu = np.asarray(psu)
    codes, inv = np.unique(psu, return_inverse=True)
    idx_by = np.split(np.argsort(inv), np.cumsum(np.bincount(inv))[:-1])
    vals = []
    for _ in range(B):
        pick = rng.integers(0, len(codes), len(codes))
        idx = np.concatenate([idx_by[k] for k in pick])
        try:
            vals.append(fn(y[idx], p[idx], w[idx]))
        except ValueError:
            pass
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def net_benefit(y, p, w, thresholds):
    W_ = np.sum(w)
    out = []
    for t in thresholds:
        pos = p >= t
        tp = np.sum(w[pos & (y == 1)]) / W_
        fp = np.sum(w[pos & (y == 0)]) / W_
        out.append(tp - fp * t / (1 - t))
    return np.array(out)


def prob_true_stunted(haz_obs, sigma_z):
    """Measurement-error model: true HAZ ~ N(haz_obs, sigma_z^2) (flat prior).
    Returns P(true HAZ < -2 | observed)."""
    return norm.cdf((STUNT_CUT - haz_obs) / sigma_z)


In [ ]:
import sys; sys.path.insert(0, PROJ + '/scripts'); os.chdir(PROJ + '/scripts')

## Stage 1 - ingestion, HAZ (WHO LMS), leakage audit
`01_prepare_data.py`

In [ ]:
%%writefile {PROJ}/scripts/01_prepare_data.py
"""
01_prepare_data.py
Stage 1 - Data ingestion, audit, outcome construction (WHO LMS HAZ),
leakage audit and locked predictor set.

Outputs:
  data/analytic.pkl                 analytic dataset (features + design + outcome)
  results/T_flow_exclusion.csv      sample flow (for flow diagram)
  results/T_leakage_audit.csv       every raw column -> role / decision / reason
  results/T_feature_domains.csv     retained predictors by domain
  results/T_sample_profile.csv      weighted profile of analytic sample
"""
import re
import numpy as np
import pandas as pd
from common import *

np.random.seed(SEED)

# ------------------------------------------------------------------ load
if os.path.exists(RAW_CACHE):
    raw = pd.read_pickle(RAW_CACHE)
else:
    raw = pd.read_excel(RAW_XLSX)
    raw.to_pickle(RAW_CACHE)
print("raw shape", raw.shape)
df = raw.copy()

C_H = "J02.b.Tinggi/Panjang Badan (cm)"
C_POS = "J02.c.KHUSUS UNTUK BALITA, (Posisi pengukuran TB/PB)"
C_WT = "J01.c.Berat Badan (kg)"
C_SEX = "4. Jenis Kelamin"
C_DOB = "6. Tanggal Lahir"
C_DOI = "2. Tanggal Pengumpulan data: (tgl-bln)"
C_PROV = "1.Provinsi"
C_KAB = "2. Kabupaten/Kota"


def parse_dmy(x):
    s = x.astype("Int64").astype(str).str.zfill(8)
    return pd.to_datetime(s, format="%d%m%Y", errors="coerce")


flow = [("Raw SKI 2023 under-five records", len(df))]

# ------------------------------------------------------------------ age
dob, doi = parse_dmy(df[C_DOB]), parse_dmy(df[C_DOI])
df["age_days"] = (doi - dob).dt.days
# fallback to reported completed months when dates are invalid
fallback = (df["7. Umur Bulan"] * 30.4375 + df["7. Umur hari"]).round()
bad_date = df["age_days"].isna() | (df["age_days"] < 0) | (df["age_days"] > 1856)
df.loc[bad_date, "age_days"] = fallback[bad_date]
df["age_months"] = df["age_days"] / 30.4375
df["sex"] = df[C_SEX].map({"Laki-laki": 1, "Prempuan": 2})

m = df["age_days"].between(0, 1856)
df = df[m]; flow.append(("Age 0-59 months (0-1856 days)", len(df)))

m = df[C_H].notna() & (df[C_H] > 0)
df = df[m]; flow.append(("Length/height measured", len(df)))

# ------------------------------------------------------------------ WHO position adjustment
h = df[C_H].astype(float).copy()
pos = df[C_POS]
standing_u2 = (df["age_days"] < 731) & (pos == "Berdiri")
lying_o2 = (df["age_days"] >= 731) & (pos == "Telentang")
h[standing_u2] += POSITION_BIAS_CM
h[lying_o2] -= POSITION_BIAS_CM
df["height_adj"] = h
df["position_mismatch"] = (standing_u2 | lying_o2).astype(int)

lms = load_lms()
L, M, S = lms_lookup(lms, df["sex"], df["age_days"])
df["lms_L"], df["lms_M"], df["lms_S"] = L, M, S
df["haz"] = haz_from_height(df["height_adj"].values, L, M, S)
df["z_per_cm"] = z_per_cm(M, S)

m = df["haz"].between(-6, 6)
flow.append(("Excluded biologically implausible HAZ (|HAZ|>6, WHO flag)", int((~m).sum())))
df = df[m]; flow.append(("Valid HAZ", len(df)))

m = df["Penimbang Populasi Individu"].notna() & df["Primary Sampling Unit"].notna()
df = df[m]; flow.append(("Complete survey design (weight, PSU, strata)", len(df)))

m = ~df["ID ART"].duplicated()
df = df[m]; flow.append(("Final analytic sample (unique child ID)", len(df)))

df["stunted"] = (df["haz"] < STUNT_CUT).astype(int)
df[W] = df["Penimbang Populasi Individu"].astype(float)
df[PSU] = df["Primary Sampling Unit"].astype(np.int64)
df[STRATA] = df["STRATA"].astype(np.int64)
df["province"] = df[C_PROV]
df["urban"] = (df["5. Klasifikasi Desa/Kelurahan"] == "Perkotaan").astype(int)

# ------------------------------------------------------------------ sentinel cleaning
SENT = {
    "H01.Berapa umur [NAMA] ketika pertama kali hamil?": [88, 98],
    "I04.Usia kehamilan saat [NAMA] dilahirkan": [88, 98],
    "I05.a.\tBerapa berat badan [NAMA] saat dilahirkan": [888, 8888, 9999],
    "I07.Berapa panjang badan [NAMA] saat dilahirkan": [0, 88, 88.8, 888],
    "I10.Salin dari catatan/dokumen lingkar kepala [NAMA]": [88, 88.8],
    "1.a.Waktu yang diperlukan dari Rumah ke faskes - Puskesmas": [995, 996, 997, 998, 999],
}
for c, codes in SENT.items():
    df.loc[df[c].isin(codes), c] = np.nan
for c in df.columns:
    if c.startswith(("150.a", "I50.e.Jika", "I50.f.Jika", "I50.h.Jika")):
        df.loc[df[c] == 88, c] = np.nan
# implausible birth weight / length
bw = "I05.a.\tBerapa berat badan [NAMA] saat dilahirkan"
bl = "I07.Berapa panjang badan [NAMA] saat dilahirkan"
df.loc[~df[bw].between(500, 6000), bw] = np.nan
df.loc[~df[bl].between(30, 60), bl] = np.nan

# ------------------------------------------------------------------ leakage audit
def domain_of(c):
    if c in ("age_months", "sex") or c.startswith(("I15.", "A12", "A22", "B01", "B07", "B16", "J01.b", "I16")):
        return "Child"
    if c.startswith(("I04", "I05", "I07", "I09", "I10", "I12", "I01", "H09", "H29", "H32", "H01")):
        return "Birth, ANC & neonatal care"
    if c.startswith(("I37", "I50", "150.")):
        return "Infant & young child feeding"
    if c.startswith(("8. Pendidikan", "9. Status", "11. Kepemilikan", "G0", "G1", "G2", "G3", "J03", "J07")):
        return "Maternal / caregiver"
    if c.startswith(("1.a.", "1. Apakah [RUMAH")):
        return "Health-service access"
    if re.match(r"^(1\.Apakah jenis|6\.|6\.b|7\.|8\.Apakah|11\.|12\.|14\.|17\.|18\.|19\.|20\.|21\.)", c):
        return "WASH"
    if c.startswith(("25.", "26.", "27.", "28.", "2.Luas", "urban")):
        return "Housing & area"
    if c.startswith(("1. Banyaknya", "2. Banyaknya", "3. Banyaknya", "4. Banyaknya", "5. Banyaknya",
                     "3. Hubungan", "1.Apakah ada anggota")):
        return "Household demography"
    return "Other"


rows = []
EXCL = {}
# Tier 1: deterministic outcome inputs / outcome
for c in [C_H, C_POS, "haz", "stunted", "height_adj", "position_mismatch", "z_per_cm",
          "lms_L", "lms_M", "lms_S"]:
    EXCL[c] = ("T1 Deterministic outcome component", "Defines HAZ / stunting label")
# Tier 2: concurrent child anthropometry
EXCL[C_WT] = ("T2 Concurrent child anthropometry", "Child weight measured with height; strongly collinear with length")
EXCL["J01.a"] = EXCL[C_WT]
# Tier 3: post-outcome proxies (nutrition-status driven programme)
for c in df.columns:
    if c.startswith("I49"):
        EXCL[c] = ("T3 Post-outcome proxy", "Supplementary feeding given BECAUSE of nutritional status")
    if c.startswith(("G03.b", "G03.2", "G03.3", "G03.4", "G03.5", "G03.6", "G03.7", "G03.8", "G03.9", "G03.c")):
        EXCL[c] = ("T4 Low-quality / conditional item", ">50% structurally missing free-text-like knowledge item")
# Tier 4 identifiers & design
for c in ["ID ART", "ID RT", "ID Ibu", "ID Anak Terakhir", "Penimbang Populasi Individu",
          "Penimbang Populasi RT", "Primary Sampling Unit", "STRATA", W, PSU, STRATA,
          C_PROV, C_KAB, "province", C_DOB, C_DOI, "age_days", "7. Umur Bulan", "7. Umur hari",
          "Kode umur", "1. No Urut ART", C_SEX, "5. Klasifikasi Desa/Kelurahan"]:
    EXCL[c] = ("Design / identifier", "Used for weighting, clustering, splitting or recoded")
EXCL["age_days"] = ("Design / identifier", "Recoded to age_months")
EXCL[C_SEX] = ("Design / identifier", "Recoded to sex")
EXCL["5. Klasifikasi Desa/Kelurahan"] = ("Design / identifier", "Recoded to urban")

cand = [c for c in df.columns]
feat = []
for c in cand:
    miss = df[c].isna().mean() if c in df else np.nan
    if c in EXCL:
        role, reason = EXCL[c]; keep = False
    elif df[c].dtype == object and df[c].nunique() > 50:
        role, reason, keep = "Excluded", "Free text (>50 levels)", False
    elif miss > 0.95:
        role, reason, keep = "Excluded", "Missing >95%", False
    elif df[c].nunique(dropna=True) <= 1:
        role, reason, keep = "Excluded", "Constant", False
    else:
        role, reason, keep = "Retained predictor", "Available before anthropometric confirmation", True
    note = ""
    if c in (bl, bw, "I10.Salin dari catatan/dokumen lingkar kepala [NAMA]"):
        note = "Birth record (pre-outcome); sensitivity model without birth anthropometry"
    if c.startswith(("J03", "J07")):
        note = "Maternal measurement (values adult-range, r(age)~0), NOT child"
    rows.append(dict(column=c, role=role, reason=reason, retained=keep,
                     domain=domain_of(c) if keep else "", missing=round(float(miss), 4), note=note))
    if keep:
        feat.append(c)

audit = pd.DataFrame(rows)
audit.to_csv(os.path.join(RES_DIR, "T_leakage_audit.csv"), index=False)
dom = audit[audit.retained].groupby("domain").size().rename("n_features").reset_index()
dom.to_csv(os.path.join(RES_DIR, "T_feature_domains.csv"), index=False)
print(dom)
print("excluded by role:\n", audit[~audit.retained].role.value_counts())

# ------------------------------------------------------------------ save analytic
keep_cols = feat + ["haz", "stunted", "height_adj", C_H, C_POS, "position_mismatch", "age_days",
                    "z_per_cm", "lms_L", "lms_M", "lms_S", W, PSU, STRATA, "province", "ID ART"]
keep_cols = list(dict.fromkeys(keep_cols))
an = df[keep_cols].copy()
an = an.rename(columns={C_H: "height_raw", C_POS: "position"})
an.attrs["features"] = feat
pd.to_pickle({"df": an, "features": feat, "birth_anthro": [bl, bw, "I10.Salin dari catatan/dokumen lingkar kepala [NAMA]"]}, ANALYTIC)

pd.DataFrame(flow, columns=["step", "n"]).to_csv(os.path.join(RES_DIR, "T_flow_exclusion.csv"), index=False)

# ------------------------------------------------------------------ profile
w = an[W].values
prof = []
def add(lbl, mask):
    prof.append(dict(characteristic=lbl, n=int(mask.sum()),
                     weighted_pct=100 * w[mask].sum() / w.sum(),
                     stunting_w_pct=100 * np.average(an.stunted[mask], weights=w[mask])))
add("All", np.ones(len(an), bool))
for lo, hi, lbl in [(0, 6, "0-5 mo"), (6, 24, "6-23 mo"), (24, 60, "24-59 mo")]:
    add("Age " + lbl, an.age_months.between(lo, hi, inclusive="left").values)
add("Boys", (an.sex == 1).values); add("Girls", (an.sex == 2).values)
add("Urban", (an.urban == 1).values); add("Rural", (an.urban == 0).values)
add("Position mismatch corrected", (an.position_mismatch == 1).values)
pd.DataFrame(prof).round(2).to_csv(os.path.join(RES_DIR, "T_sample_profile.csv"), index=False)
print(pd.DataFrame(prof).round(2))
print(pd.DataFrame(flow))
print("n features", len(feat), "| unweighted prev", an.stunted.mean(), "| weighted",
      np.average(an.stunted, weights=w))


In [ ]:
!cd "{PROJ}/scripts" && SKI_PROJECT_DIR="{PROJ}" python 01_prepare_data.py

## Stage 2 - survey-aware ML risk model & calibration (Axis B)
`02_ml_screening.py`

In [ ]:
%%writefile {PROJ}/scripts/02_ml_screening.py
"""
02_ml_screening.py
Stage 2 - Survey-aware, leakage-free risk model (Axis B of the framework).

 * spatial holdout (3 provinces) + PSU-grouped locked test set (20% PSUs)
 * 5-fold PSU-grouped CV on the development set -> out-of-fold (OOF) predictions
 * models: prevalence baseline, LR (unweighted / survey-weighted),
           RF (weighted), HGB (unweighted / weighted)
 * comparators: leaky model (child weight/height), random (non-grouped) CV,
                model without birth anthropometry
 * post-hoc calibration fitted on OOF only (Platt, isotonic), frozen for test/holdout
 * risk-tier threshold fixed on development OOF (weighted Youden J)

Outputs: results/T_model_*.csv, results/pred_*.pkl, figures/F_*.png
"""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_curve
from common import *
from plot_style import apply_style, COLORBLIND_SAFE_PALETTE as PAL

apply_style(10)
t0 = time.time()
A = pd.read_pickle(ANALYTIC)
df, FEAT, BIRTH = A["df"].reset_index(drop=True).copy(), A["features"], A["birth_anthro"]
CAT = [c for c in FEAT if df[c].dtype == object or str(df[c].dtype).startswith("str")]
NUM = [c for c in FEAT if c not in CAT]
for c in CAT:
    df[c] = df[c].astype(object)
print(f"features {len(FEAT)} (cat {len(CAT)}, num {len(NUM)})")

# ------------------------------------------------------------------ splits
hold = df.province.isin(SPATIAL_HOLDOUT).values
rest = np.where(~hold)[0]
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=SEED)
tr_i, te_i = next(gss.split(rest, groups=df[PSU].values[rest]))
split = np.array(["holdout"] * len(df), dtype=object)
split[rest[tr_i]] = "dev"; split[rest[te_i]] = "test"
df["split"] = split
assert not set(df.loc[df.split == "dev", PSU]) & set(df.loc[df.split == "test", PSU])
spl = df.groupby("split").agg(n=("stunted", "size"), n_psu=(PSU, "nunique"),
                               prev_unw=("stunted", "mean"))
spl["prev_w"] = df.groupby("split").apply(lambda g: np.average(g.stunted, weights=g[W]))
spl.to_csv(os.path.join(RES_DIR, "T_split_summary.csv")); print(spl)

dev = df[df.split == "dev"].reset_index(drop=True)
tst = df[df.split == "test"].reset_index(drop=True)
hol = df[df.split == "holdout"].reset_index(drop=True)


# ------------------------------------------------------------------ model factory
def pre_linear(num, cat):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                          ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), cat)])


def pre_tree(num, cat):
    return ColumnTransformer([
        ("num", "passthrough", num),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                               encoded_missing_value=np.nan), cat)])


def make(name, num, cat):
    if name.startswith("LR"):
        return Pipeline([("pre", pre_linear(num, cat)),
                         ("clf", LogisticRegression(C=0.1, max_iter=3000))])
    if name.startswith("RF"):
        return Pipeline([("pre", pre_linear(num, cat)),
                         ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                                        max_features="sqrt", n_jobs=-1,
                                                        random_state=SEED))])
    if name.startswith("HGB"):
        catmask = [False] * len(num) + [True] * len(cat)
        return Pipeline([("pre", pre_tree(num, cat)),
                         ("clf", HistGradientBoostingClassifier(
                             max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                             min_samples_leaf=50, l2_regularization=1.0,
                             categorical_features=catmask, early_stopping=False,
                             random_state=SEED))])
    raise ValueError(name)


def fit(model, X, y, w):
    model.fit(X, y, clf__sample_weight=None if w is None else w / w.mean())
    return model


MODELS = {  # name: (weighted?, features)
    "LR (unweighted)": (False, FEAT),
    "LR (survey-weighted)": (True, FEAT),
    "RF (survey-weighted)": (True, FEAT),
    "HGB (unweighted)": (False, FEAT),
    "HGB (survey-weighted)": (True, FEAT),
}
LEAKY_EXTRA = ["child_weight_kg", "height_raw"]
dev["child_weight_kg"] = np.nan  # placeholder, filled below from raw
raw = pd.read_pickle(RAW_CACHE).set_index("ID ART")["J01.c.Berat Badan (kg)"]
for d in (df, dev, tst, hol):
    d["child_weight_kg"] = d["ID ART"].map(raw).values

COMPARATORS = {
    "HGB weighted + child height & weight (LEAKY)": (True, FEAT + LEAKY_EXTRA, "grouped"),
    "HGB weighted, random (non-grouped) CV": (True, FEAT, "random"),
    "HGB weighted, without birth anthropometry": (True, [f for f in FEAT if f not in BIRTH], "grouped"),
}


import joblib, hashlib
CACHE = os.path.join(DATA_DIR, "cache"); os.makedirs(CACHE, exist_ok=True)


def oof_predict(name, weighted, feats, scheme="grouped"):
    key = hashlib.md5(f"{name}|{weighted}|{scheme}|{len(feats)}|{sorted(feats)}".encode()).hexdigest()[:12]
    fp = os.path.join(CACHE, f"oof_{key}.joblib")
    if os.path.exists(fp):
        return joblib.load(fp)
    res = _oof_predict(name, weighted, feats, scheme)
    joblib.dump(res, fp)
    return res


def _oof_predict(name, weighted, feats, scheme="grouped"):
    num = [c for c in feats if c not in CAT]; cat = [c for c in feats if c in CAT]
    X, y, w, g = dev[feats], dev.stunted.values, dev[W].values, dev[PSU].values
    oof = np.zeros(len(dev))
    cv = GroupKFold(n_splits=N_FOLDS) if scheme == "grouped" else KFold(N_FOLDS, shuffle=True, random_state=SEED)
    it = cv.split(X, y, groups=g) if scheme == "grouped" else cv.split(X, y)
    for k, (a, b) in enumerate(it):
        m = fit(make(name, num, cat), X.iloc[a], y[a], w[a] if weighted else None)
        oof[b] = m.predict_proba(X.iloc[b])[:, 1]
    full = fit(make(name, num, cat), X, y, w if weighted else None)
    return oof, full


def platt_fit(p, y, w):
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    X = np.column_stack([np.ones_like(lp), lp])
    b = __import__("common")._wlogreg(X, y.astype(float), w)
    return lambda q: 1 / (1 + np.exp(-(b[0] + b[1] * np.log(np.clip(q, 1e-6, 1 - 1e-6) / (1 - np.clip(q, 1e-6, 1 - 1e-6))))))


def iso_fit(p, y, w):
    ir = IsotonicRegression(out_of_bounds="clip", y_min=1e-4, y_max=1 - 1e-4).fit(p, y, sample_weight=w)
    return ir.predict


rows, preds = [], {}
yd, wd = dev.stunted.values, dev[W].values
for name, (weighted, feats) in MODELS.items():
    t = time.time()
    oof, full = oof_predict(name, weighted, feats)
    pt, ph = full.predict_proba(tst[feats])[:, 1], full.predict_proba(hol[feats])[:, 1]
    cal_p, cal_i = platt_fit(oof, yd, wd), iso_fit(oof, yd, wd)
    preds[name] = dict(oof=oof, test=pt, hold=ph, model=full, platt=cal_p, iso=cal_i)
    for part, y_, w_, p_ in [("OOF (dev)", yd, wd, oof), ("Locked test", tst.stunted.values, tst[W].values, pt),
                             ("Spatial holdout", hol.stunted.values, hol[W].values, ph)]:
        for cal, f in [("raw", lambda x: x), ("Platt", cal_p), ("isotonic", cal_i)]:
            if part == "OOF (dev)" and cal != "raw":
                continue  # calibrators are fitted on OOF: do not report in-sample
            rows.append(dict(model=name, partition=part, calibration=cal, **all_metrics(y_, f(p_), w_)))
    print(f"{name:28s} OOF AUROC {w_auc(yd, oof, wd):.4f}  ({time.time()-t:.0f}s)")

# prevalence baseline
prev = np.average(yd, weights=wd)
for part, d in [("Locked test", tst), ("Spatial holdout", hol)]:
    rows.append(dict(model="Prevalence baseline", partition=part, calibration="raw",
                     **all_metrics(d.stunted.values, np.full(len(d), prev) + 1e-9 * np.random.rand(len(d)), d[W].values)))

comp_rows = []
for name, (weighted, feats, scheme) in COMPARATORS.items():
    t = time.time()
    oof, full = oof_predict("HGB", weighted, feats, scheme)
    pt = full.predict_proba(tst[feats])[:, 1]
    comp_rows.append(dict(model=name, cv_scheme=scheme, oof_auroc=w_auc(yd, oof, wd),
                          test_auroc=w_auc(tst.stunted.values, pt, tst[W].values),
                          optimism=w_auc(yd, oof, wd) - w_auc(tst.stunted.values, pt, tst[W].values)))
    print(f"{name:45s} OOF {comp_rows[-1]['oof_auroc']:.4f} test {comp_rows[-1]['test_auroc']:.4f} ({time.time()-t:.0f}s)")
ref = [r for r in rows if r["model"] == "HGB (survey-weighted)" and r["partition"] in ("OOF (dev)",)][0]
reft = [r for r in rows if r["model"] == "HGB (survey-weighted)" and r["partition"] == "Locked test" and r["calibration"] == "raw"][0]
comp_rows.insert(0, dict(model="HGB weighted, PSU-grouped CV (proposed)", cv_scheme="grouped",
                         oof_auroc=ref["auroc"], test_auroc=reft["auroc"], optimism=ref["auroc"] - reft["auroc"]))
pd.DataFrame(comp_rows).round(4).to_csv(os.path.join(RES_DIR, "T_leakage_and_cv_comparators.csv"), index=False)

M = pd.DataFrame(rows)
M.round(4).to_csv(os.path.join(RES_DIR, "T_model_metrics_all.csv"), index=False)

# ------------------------------------------------------------------ bootstrap CI on locked test (Platt)
ci_rows = []
for name in MODELS:
    p = preds[name]["platt"](preds[name]["test"])
    y_, w_, g_ = tst.stunted.values, tst[W].values, tst[PSU].values
    a_lo, a_hi = psu_bootstrap(y_, p, w_, g_, w_auc)
    e_lo, e_hi = psu_bootstrap(y_, p, w_, g_, w_ece)
    ci_rows.append(dict(model=name, auroc=w_auc(y_, p, w_), auroc_lo=a_lo, auroc_hi=a_hi,
                        ece=w_ece(y_, p, w_), ece_lo=e_lo, ece_hi=e_hi,
                        brier=w_brier(y_, p, w_), auprc=w_auprc(y_, p, w_)))
CI = pd.DataFrame(ci_rows); CI.round(4).to_csv(os.path.join(RES_DIR, "T_model_comparison_test_CI.csv"), index=False)
print(CI.round(4))

# ------------------------------------------------------------------ select final model
# pre-specified rule: highest survey-weighted OOF AUROC on the development set
oof_auc = {n: w_auc(yd, preds[n]["oof"], wd) for n in MODELS}
BEST = max(oof_auc, key=oof_auc.get)
print("selected model:", BEST, oof_auc)
calP = preds[BEST]["platt"]
p_oof_cal = calP(preds[BEST]["oof"])
# risk threshold fixed on development OOF: weighted Youden J
fpr, tpr, thr = roc_curve(yd, p_oof_cal, sample_weight=wd)
j = tpr - fpr; T_RISK = float(thr[np.argmax(j)])
save_json(dict(best_model=BEST, risk_threshold=T_RISK, prevalence_dev_w=prev), os.path.join(RES_DIR, "config_locked.json"))
print("locked risk threshold", T_RISK)

out = {}
for nm, d, key in [("dev", dev, "oof"), ("test", tst, "test"), ("hold", hol, "hold")]:
    o = d[["ID ART", "haz", "stunted", "height_adj", "height_raw", "position", "position_mismatch",
           "age_days", "age_months", "sex", "urban", "z_per_cm", "lms_M", "lms_S", "lms_L", W, PSU, "province"]].copy()
    o["p_raw"] = preds[BEST][key]; o["p_cal"] = calP(o["p_raw"].values)
    out[nm] = o
pd.to_pickle(out, os.path.join(DATA_DIR, "predictions.pkl"))

# threshold performance
thr_rows = []
for nm in ("test", "hold"):
    o = out[nm]; y_, w_ = o.stunted.values, o[W].values; hi = o.p_cal.values >= T_RISK
    sens = np.sum(w_[hi & (y_ == 1)]) / np.sum(w_[y_ == 1]); spec = np.sum(w_[~hi & (y_ == 0)]) / np.sum(w_[y_ == 0])
    ppv = np.sum(w_[hi & (y_ == 1)]) / np.sum(w_[hi]); npv = np.sum(w_[~hi & (y_ == 0)]) / np.sum(w_[~hi])
    thr_rows.append(dict(partition=nm, threshold=T_RISK, flagged_high_w_pct=100 * w_[hi].sum() / w_.sum(),
                         sensitivity=sens, specificity=spec, ppv=ppv, npv=npv))
pd.DataFrame(thr_rows).round(4).to_csv(os.path.join(RES_DIR, "T_risk_threshold_performance.csv"), index=False)
print(pd.DataFrame(thr_rows).round(3))

# ------------------------------------------------------------------ subgroup analysis (test+holdout, Platt)
sub_rows = []
ev = pd.concat([out["test"].assign(part="Locked test"), out["hold"].assign(part="Spatial holdout")])
groups = {"Age 0-5 mo": ev.age_months < 6, "Age 6-23 mo": ev.age_months.between(6, 24, inclusive="left"),
          "Age 24-59 mo": ev.age_months >= 24, "Boys": ev.sex == 1, "Girls": ev.sex == 2,
          "Urban": ev.urban == 1, "Rural": ev.urban == 0,
          "Locked test": ev.part == "Locked test", "Spatial holdout": ev.part == "Spatial holdout"}
for p_ in SPATIAL_HOLDOUT:
    groups["Holdout: " + p_] = ev.province == p_
for g, m in groups.items():
    e = ev[m.values]; y_, p_, w_ = e.stunted.values, e.p_cal.values, e[W].values
    lo, hi_ = psu_bootstrap(y_, p_, w_, e[PSU].values, w_auc, B=200)
    mt = all_metrics(y_, p_, w_)
    sub_rows.append(dict(group=g, auroc_lo=lo, auroc_hi=hi_, **mt))
SUB = pd.DataFrame(sub_rows); SUB.round(4).to_csv(os.path.join(RES_DIR, "T_subgroup.csv"), index=False)
print(SUB[["group", "n", "auroc", "auroc_lo", "auroc_hi", "ece", "cal_intercept"]].round(3))

# ------------------------------------------------------------------ grouped permutation importance (domain level)
from common import w_auc as _wa
feat_dom = pd.read_csv(os.path.join(RES_DIR, "T_leakage_audit.csv")).query("retained").set_index("column").domain
model = preds[BEST]["model"]
base = _wa(tst.stunted.values, model.predict_proba(tst[FEAT])[:, 1], tst[W].values)
rng = np.random.default_rng(SEED); imp = []
for dname in sorted(feat_dom.unique()):
    cols = [c for c in FEAT if feat_dom.get(c) == dname]
    drops = []
    for r in range(5):
        Xp = tst[FEAT].copy(); perm = rng.permutation(len(Xp))
        Xp[cols] = Xp[cols].values[perm]
        drops.append(base - _wa(tst.stunted.values, model.predict_proba(Xp)[:, 1], tst[W].values))
    imp.append(dict(domain=dname, n_features=len(cols), auroc_drop=np.mean(drops), sd=np.std(drops)))
single = []
for c in FEAT:
    Xp = tst[FEAT].copy(); Xp[c] = Xp[c].values[rng.permutation(len(Xp))]
    single.append(dict(feature=c, domain=feat_dom.get(c), auroc_drop=base - _wa(tst.stunted.values, model.predict_proba(Xp)[:, 1], tst[W].values)))
IMP = pd.DataFrame(imp).sort_values("auroc_drop", ascending=False)
IMP.round(5).to_csv(os.path.join(RES_DIR, "T_importance_domain.csv"), index=False)
SI = pd.DataFrame(single).sort_values("auroc_drop", ascending=False)
SI.round(5).to_csv(os.path.join(RES_DIR, "T_importance_feature.csv"), index=False)
print(IMP.round(4)); print(SI.head(15).round(4))

# ================================================================== FIGURES
# F5 model comparison (AUROC with CI | ECE with CI)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
order = CI.sort_values("auroc").model.tolist(); c = CI.set_index("model").loc[order]
yy = np.arange(len(order))
ax[0].errorbar(c.auroc, yy, xerr=[c.auroc - c.auroc_lo, c.auroc_hi - c.auroc], fmt="o", color=PAL[0], capsize=3)
ax[0].axvline(0.5, color="#888", ls="--", lw=1); ax[0].set_yticks(yy); ax[0].set_yticklabels(order)
ax[0].set_xlabel("Weighted AUROC (95% PSU-bootstrap CI)"); ax[0].set_title("Discrimination (locked test)")
ax[1].errorbar(c.ece, yy, xerr=[c.ece - c.ece_lo, c.ece_hi - c.ece], fmt="s", color=PAL[1], capsize=3)
ax[1].set_yticks(yy); ax[1].set_yticklabels([]); ax[1].set_xlabel("Weighted ECE after Platt (95% CI)")
ax[1].set_title("Calibration (locked test)")
fig.savefig(os.path.join(FIG_DIR, "F05_model_comparison.png")); plt.close(fig)

# F6 ROC overlay + DCA
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
for k, name in enumerate(MODELS):
    f_, t_, _ = roc_curve(tst.stunted.values, preds[name]["test"], sample_weight=tst[W].values)
    ax[0].plot(f_, t_, lw=1.6, color=PAL[k], label=f"{name} ({w_auc(tst.stunted.values, preds[name]['test'], tst[W].values):.3f})")
f_, t_, _ = roc_curve(tst.stunted.values, np.zeros(len(tst)) + 1e-9 * np.random.rand(len(tst)), sample_weight=tst[W].values)
ax[0].plot([0, 1], [0, 1], "--", color="#888", lw=1)
ax[0].set_xlabel("1 - specificity"); ax[0].set_ylabel("Sensitivity"); ax[0].set_title("Weighted ROC, locked test")
ax[0].legend(fontsize=7, loc="lower right")
ts = np.linspace(0.05, 0.60, 56)
yt, wt = tst.stunted.values, tst[W].values
pc = out["test"].p_cal.values
ax[1].plot(ts, net_benefit(yt, pc, wt, ts), color=PAL[0], lw=2, label=f"{BEST} + Platt")
ax[1].plot(ts, net_benefit(yt, np.ones(len(yt)), wt, ts), color=PAL[1], lw=1.5, label="Measure/act on all")
ax[1].plot(ts, np.zeros_like(ts), color="#555", lw=1.2, label="Act on none")
ax[1].axvline(T_RISK, color=PAL[2], ls=":", lw=1.5, label=f"Locked risk threshold = {T_RISK:.3f}")
ax[1].set_ylim(-0.05, 0.25); ax[1].set_xlabel("Threshold probability"); ax[1].set_ylabel("Net benefit")
ax[1].set_title("Decision curve, locked test"); ax[1].legend(fontsize=7)
fig.savefig(os.path.join(FIG_DIR, "F06_roc_dca.png")); plt.close(fig)

# F7 reliability diagram raw vs Platt vs isotonic on test and holdout, weighted bins + CI via Neff
def rel_bins(y, p, w, nb=10):
    b = np.clip((p * nb).astype(int), 0, nb - 1); r = []
    for k in range(nb):
        m = b == k
        if m.sum() < 5: continue
        ww = w[m]; neff = ww.sum() ** 2 / np.sum(ww ** 2); o = np.average(y[m], weights=ww)
        se = np.sqrt(o * (1 - o) / neff)
        r.append(dict(bin=f"[{k/nb:.1f},{(k+1)/nb:.1f})", n_raw=int(m.sum()), n_eff=neff, w_frac=ww.sum() / w.sum(),
                      mean_pred=np.average(p[m], weights=ww), obs=o, lo=max(0, o - 1.96 * se), hi=min(1, o + 1.96 * se)))
    return pd.DataFrame(r)

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
bin_tabs = []
for a, (nm, d, key) in zip(ax, [("Locked test", tst, "test"), ("Spatial holdout", hol, "hold")]):
    for k, (lbl, f) in enumerate([("Raw", lambda x: x), ("Platt", preds[BEST]["platt"]), ("Isotonic", preds[BEST]["iso"])]):
        p = f(preds[BEST][key]); rb = rel_bins(d.stunted.values, p, d[W].values)
        rb.insert(0, "calibration", lbl); rb.insert(0, "partition", nm); bin_tabs.append(rb)
        a.errorbar(rb.mean_pred, rb.obs, yerr=[rb.obs - rb.lo, rb.hi - rb.obs], fmt="-o", ms=4, lw=1.4,
                   capsize=2, color=PAL[k], label=lbl)
    a.plot([0, 1], [0, 1], "--", color="#888", lw=1); a.set_xlim(0, 0.8); a.set_ylim(0, 0.8)
    a.set_xlabel("Mean predicted probability"); a.set_ylabel("Observed stunting (weighted)"); a.set_title(nm)
    a.legend(fontsize=8)
pd.concat(bin_tabs).round(4).to_csv(os.path.join(RES_DIR, "T_calibration_bins.csv"), index=False)
fig.savefig(os.path.join(FIG_DIR, "F07_calibration.png")); plt.close(fig)

# F8 subgroup forest + domain importance
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.8))
s = SUB.iloc[::-1]; yy = np.arange(len(s))
ax[0].errorbar(s.auroc, yy, xerr=[s.auroc - s.auroc_lo, s.auroc_hi - s.auroc], fmt="o", color=PAL[0], capsize=2)
ax[0].axvline(SUB.set_index("group").loc["Locked test", "auroc"], color="#888", ls=":", lw=1)
ax[0].set_yticks(yy); ax[0].set_yticklabels([f"{g} (n={n:,})" for g, n in zip(s.group, s.n)], fontsize=7.5)
ax[0].set_xlabel("Weighted AUROC (95% CI)"); ax[0].set_title("Subgroup discrimination")
im = IMP.iloc[::-1]
ax[1].barh(im.domain, im.auroc_drop, xerr=im.sd, color=PAL[2], height=0.6)
ax[1].set_xlabel("AUROC drop (domain permuted)"); ax[1].set_title("Domain importance (test)")
ax[1].tick_params(axis="y", labelsize=8)
fig.savefig(os.path.join(FIG_DIR, "F08_subgroup_importance.png")); plt.close(fig)

print(f"done in {time.time()-t0:.0f}s")


In [ ]:
!cd "{PROJ}/scripts" && SKI_PROJECT_DIR="{PROJ}" python 02_ml_screening.py

## Stage 3 - measurement uncertainty & re-measurement policies (Axis A)
`03_measurement_uncertainty.py`

In [ ]:
%%writefile {PROJ}/scripts/03_measurement_uncertainty.py
"""
03_measurement_uncertainty.py
Stage 3 - Anthropometric measurement uncertainty (Axis A of the framework).

 A. How large is 1 cm of length/height error in HAZ units, by age and sex? (WHO LMS)
 B. Deterministic perturbation stress test in CENTIMETRES (+/-0.5, +/-1.0 cm and
    +/-0.7 cm recumbent/standing position error) vs the draft's HAZ-shift test.
 C. Monte Carlo simulation of random measurement error (sigma = 0.5/1.0/1.5 cm):
    how many children would receive the wrong stunting label?
 D. Re-measurement policy comparison: which children should be re-measured?
    P0 none | P1 fixed band HAZ -2 +/- 0.10 SD (draft) | P2 fixed band tuned to the
    same budget as P3 | P3 proposed per-child probability band
    alpha < P(true HAZ < -2 | observed) < 1 - alpha.

Simulation truth = recorded (position-corrected) height; simulated observation =
truth + N(0, sigma^2). Survey weights are used for all population estimates.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from common import *
from plot_style import apply_style, COLORBLIND_SAFE_PALETTE as PAL

apply_style(10)
rng = np.random.default_rng(SEED)
A = pd.read_pickle(ANALYTIC); d = A["df"].reset_index(drop=True)
h, M, S, w = d.height_adj.values, d.lms_M.values, d.lms_S.values, d[W].values
age = d.age_months.values; zpc = d.z_per_cm.values; haz = d.haz.values
y_true = (haz < STUNT_CUT).astype(int)
AGEG = np.select([age < 6, age < 24], ["0-5 mo", "6-23 mo"], "24-59 mo")
GROUPS = ["0-5 mo", "6-23 mo", "24-59 mo"]
wp = lambda m: 100 * w[m].sum() / w.sum()


def zfromh(hh):
    return (hh / M - 1.0) / S


# ------------------------------------------------------------------ A. z per cm
lms = load_lms()
lms["m_months"] = lms.age / 30.4375
lms["sd_per_cm"] = 1 / (lms.m * lms.s)
lms["cm_per_0.10SD"] = 0.10 * lms.m * lms.s
tabA = []
for mo in [0, 3, 6, 12, 18, 24, 36, 48, 59]:
    r = lms[lms.age == int(round(mo * 30.4375))]
    for sx, lab in [(1, "Boys"), (2, "Girls")]:
        q = r[r.sex == sx].iloc[0]
        tabA.append(dict(age_months=mo, sex=lab, median_cm=q.m, sd_cm=q.m * q.s,
                         SD_per_1cm=q.sd_per_cm, cm_equiv_0p10SD=q["cm_per_0.10SD"]))
TA = pd.DataFrame(tabA).round(3); TA.to_csv(os.path.join(RES_DIR, "T_sd_per_cm_by_age.csv"), index=False)
print(TA)

# ------------------------------------------------------------------ B. deterministic stress test
rowsB = []
scen = [("Height -1.0 cm", -1.0, None), ("Height -0.5 cm", -0.5, None), ("Height +0.5 cm", 0.5, None),
        ("Height +1.0 cm", 1.0, None), ("Position error -0.7 cm", -0.7, None), ("Position error +0.7 cm", 0.7, None),
        ("Draft: HAZ -0.10 SD", None, -0.10), ("Draft: HAZ -0.05 SD", None, -0.05),
        ("Draft: HAZ +0.05 SD", None, 0.05), ("Draft: HAZ +0.10 SD", None, 0.10)]
for lbl, dcm, dz in scen:
    z2 = zfromh(h + dcm) if dcm is not None else haz + dz
    sw = ((z2 < STUNT_CUT).astype(int) != y_true)
    row = dict(scenario=lbl, overall_switch_pct=wp(sw),
               prevalence_after_pct=100 * np.average(z2 < STUNT_CUT, weights=w),
               n_switched=int(sw.sum()))
    for g in GROUPS:
        m = AGEG == g
        row[f"switch_{g}_pct"] = 100 * w[m & sw].sum() / w[m].sum()
    rowsB.append(row)
TB = pd.DataFrame(rowsB).round(3); TB.to_csv(os.path.join(RES_DIR, "T_perturbation_cm_vs_haz.csv"), index=False)
print(TB)
prev0 = 100 * np.average(y_true, weights=w)

# ------------------------------------------------------------------ C/D. Monte Carlo
R = 100
SIGMAS = SIGMA_CM_SCENARIOS
KGRID = np.round(np.arange(0.02, 0.62, 0.02), 2)          # fixed band half-widths (SD)
AGRID = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40]  # proposed alpha grid


def evaluate(flag, obs_lab, h_obs, sig):
    """Remeasure flagged children once (independent error) and average the two readings."""
    err = obs_lab != y_true
    h2 = h + rng.normal(0, sig, len(h))
    lab2 = (zfromh((h_obs + h2) / 2) < STUNT_CUT).astype(int)
    final = np.where(flag, lab2, obs_lab)
    return dict(burden=wp(flag), err_before=wp(err), capture=100 * w[flag & err].sum() / max(w[err].sum(), 1e-9),
                err_after=wp(final != y_true), yield_pct=100 * w[flag & err].sum() / max(w[flag].sum(), 1e-9))


mc_rows, curve_rows = [], []
for sig in SIGMAS:
    for r in range(R):
        h_obs = h + rng.normal(0, sig, len(h))
        z_obs = zfromh(h_obs); lab = (z_obs < STUNT_CUT).astype(int); err = lab != y_true
        # C. misclassification
        row = dict(sigma_cm=sig, rep=r, misclass_pct=wp(err),
                   false_stunted_pct=wp(err & (lab == 1)), missed_stunted_pct=wp(err & (lab == 0)),
                   prev_obs_pct=100 * np.average(lab, weights=w))
        for g in GROUPS:
            m = AGEG == g; row[f"misclass_{g}_pct"] = 100 * w[m & err].sum() / w[m].sum()
        mc_rows.append(row)
        if r >= 30:
            continue  # policy curves use first 30 replicates (stable to 3 d.p.)
        pst = norm.cdf((STUNT_CUT - z_obs) / (sig * zpc))
        for k in KGRID:
            f = np.abs(z_obs - STUNT_CUT) <= k
            curve_rows.append(dict(sigma_cm=sig, rep=r, policy="Fixed HAZ band", param=k, **evaluate(f, lab, h_obs, sig)))
        for a in AGRID:
            f = (pst > a) & (pst < 1 - a)
            curve_rows.append(dict(sigma_cm=sig, rep=r, policy="Proposed probability band", param=a, **evaluate(f, lab, h_obs, sig)))
        curve_rows.append(dict(sigma_cm=sig, rep=r, policy="No re-measurement", param=0,
                               **evaluate(np.zeros(len(h), bool), lab, h_obs, sig)))
    print("sigma", sig, "done")

MC = pd.DataFrame(mc_rows)
TC = MC.groupby("sigma_cm").agg(["mean", lambda x: np.percentile(x, 2.5), lambda x: np.percentile(x, 97.5)])
TC.columns = [f"{a}_{'mean' if b == 'mean' else ('lo' if b == '<lambda_0>' else 'hi')}" for a, b in TC.columns]
TC = TC.drop(columns=[c for c in TC.columns if c.startswith("rep_")]).round(3)
TC.to_csv(os.path.join(RES_DIR, "T_montecarlo_misclassification.csv")); print(TC.T)

CV = pd.DataFrame(curve_rows).groupby(["sigma_cm", "policy", "param"]).mean(numeric_only=True).drop(columns="rep").reset_index()
CV.round(3).to_csv(os.path.join(RES_DIR, "T_policy_curves.csv"), index=False)

# head-to-head table at sigma = 1.0 cm: draft band vs proposed (alpha=0.10) vs fixed band with equal burden
H = []
for sig in SIGMAS:
    c = CV[CV.sigma_cm == sig]
    none = c[c.policy == "No re-measurement"].iloc[0]
    draft = c[(c.policy == "Fixed HAZ band") & (np.isclose(c.param, 0.10))].iloc[0]
    prop = c[(c.policy == "Proposed probability band") & (np.isclose(c.param, ALPHA_MAIN))].iloc[0]
    fb = c[c.policy == "Fixed HAZ band"]; eq = fb.iloc[(fb.burden - prop.burden).abs().argmin()]
    for lbl, rr in [("P0 No re-measurement (hard threshold)", none), ("P1 Draft fixed band HAZ -2 +/- 0.10 SD", draft),
                    (f"P2 Fixed band +/- {eq.param:.2f} SD (budget-matched to P3)", eq),
                    (f"P3 Proposed per-child band (alpha = {ALPHA_MAIN})", prop)]:
        H.append(dict(sigma_cm=sig, policy=lbl, remeasure_burden_pct=rr.burden, errors_captured_pct=rr.capture,
                      yield_pct=rr.yield_pct, misclass_before_pct=rr.err_before, misclass_after_pct=rr.err_after,
                      error_reduction_pct=100 * (rr.err_before - rr.err_after) / rr.err_before))
TH = pd.DataFrame(H).round(3); TH.to_csv(os.path.join(RES_DIR, "T_policy_head_to_head.csv"), index=False)
print(TH.to_string())

# robustness: proposed rule built with sigma_assumed = 1.0 while true sigma differs
rob = []
for sig in SIGMAS:
    for r in range(20):
        h_obs = h + rng.normal(0, sig, len(h)); z_obs = zfromh(h_obs); lab = (z_obs < STUNT_CUT).astype(int)
        pst = norm.cdf((STUNT_CUT - z_obs) / (SIGMA_CM_MAIN * zpc))
        f = (pst > ALPHA_MAIN) & (pst < 1 - ALPHA_MAIN)
        rob.append(dict(true_sigma=sig, assumed_sigma=SIGMA_CM_MAIN, **evaluate(f, lab, h_obs, sig)))
TR = pd.DataFrame(rob).groupby(["true_sigma", "assumed_sigma"]).mean().reset_index().round(3)
TR.to_csv(os.path.join(RES_DIR, "T_policy_sigma_misspecification.csv"), index=False); print(TR)

# age-specific capture at sigma = 1.0 cm: fixed band vs proposed at equal burden
ag = []
k_eq = float(TH[(TH.sigma_cm == SIGMA_CM_MAIN) & TH.policy.str.startswith("P2")].policy.str.extract(r"([0-9.]+) SD")[0].iloc[0])
for r in range(30):
    h_obs = h + rng.normal(0, SIGMA_CM_MAIN, len(h)); z_obs = zfromh(h_obs); lab = (z_obs < STUNT_CUT).astype(int)
    err = lab != y_true
    pst = norm.cdf((STUNT_CUT - z_obs) / (SIGMA_CM_MAIN * zpc))
    flags = {"P1 draft +/-0.10 SD": np.abs(z_obs - STUNT_CUT) <= 0.10,
             f"P2 fixed +/-{k_eq:.2f} SD": np.abs(z_obs - STUNT_CUT) <= k_eq,
             "P3 proposed (alpha=0.10)": (pst > ALPHA_MAIN) & (pst < 1 - ALPHA_MAIN)}
    for pn, f in flags.items():
        for g in GROUPS:
            m = AGEG == g
            ag.append(dict(policy=pn, age_group=g, rep=r, burden=100 * w[m & f].sum() / w[m].sum(),
                           capture=100 * w[m & f & err].sum() / w[m & err].sum()))
TG = pd.DataFrame(ag).groupby(["policy", "age_group"]).mean(numeric_only=True).drop(columns="rep").reset_index().round(2)
TG.to_csv(os.path.join(RES_DIR, "T_policy_capture_by_age.csv"), index=False); print(TG)

# ================================================================== FIGURES
# F03: SD per cm and cm-equivalent of 0.10 SD by age
fig, ax = plt.subplots(1, 2, figsize=(9, 3.3))
for k, (sx, lab) in enumerate([(1, "Boys"), (2, "Girls")]):
    q = lms[lms.sex == sx]
    ax[0].plot(q.m_months, q.sd_per_cm, color=PAL[k], lw=2, label=lab)
    ax[1].plot(q.m_months, q["cm_per_0.10SD"], color=PAL[k], lw=2, label=lab)
ax[0].set_xlabel("Age (months)"); ax[0].set_ylabel("HAZ units per 1 cm error"); ax[0].set_title("(a) Impact of a 1 cm error")
for t, ls in [(0.5, ":"), (1.0, "--")]:
    ax[1].axhline(t, color="#555", ls=ls, lw=1)
    ax[1].text(59, t + 0.02, f"field error SD {t} cm", ha="right", fontsize=7, color="#555")
ax[1].set_xlabel("Age (months)"); ax[1].set_ylabel("cm equivalent of 0.10 SD"); ax[1].set_ylim(0, 1.2)
ax[1].set_title("(b) Draft band +/-0.10 SD in cm"); ax[0].legend(); ax[1].legend(loc="center right")
fig.savefig(os.path.join(FIG_DIR, "F03_sd_per_cm.png")); plt.close(fig)

# F04: switching by scenario and age group
fig, ax = plt.subplots(figsize=(9, 3.4))
tb = TB.set_index("scenario"); x = np.arange(len(tb)); bw_ = 0.26
for k, g in enumerate(GROUPS):
    ax.bar(x + (k - 1) * bw_, tb[f"switch_{g}_pct"], width=bw_ - 0.03, color=PAL[k], label=g)
ax.set_xticks(x); ax.set_xticklabels(tb.index, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("Children whose label switches (%)"); ax.set_title("Label switching under centimetre vs HAZ perturbation")
ax.legend(title="Age")
fig.savefig(os.path.join(FIG_DIR, "F04_label_switching.png")); plt.close(fig)

# F09: Monte Carlo misclassification by sigma and age
fig, ax = plt.subplots(1, 2, figsize=(9, 3.3))
x = np.arange(len(SIGMAS))
for k, g in enumerate(GROUPS):
    ax[0].bar(x + (k - 1) * 0.26, TC[f"misclass_{g}_pct_mean"], 0.23, color=PAL[k], label=g,
              yerr=[TC[f"misclass_{g}_pct_mean"] - TC[f"misclass_{g}_pct_lo"], TC[f"misclass_{g}_pct_hi"] - TC[f"misclass_{g}_pct_mean"]], capsize=2)
ax[0].set_xticks(x); ax[0].set_xticklabels([f"{s} cm" for s in SIGMAS]); ax[0].set_xlabel("Random measurement error SD")
ax[0].set_ylabel("Wrong stunting label (%)"); ax[0].set_title("(a) Misclassification by age"); ax[0].legend(fontsize=8)
ax[1].bar(x - 0.18, TC["false_stunted_pct_mean"], 0.34, color=PAL[1], label="Falsely labelled stunted")
ax[1].bar(x + 0.18, TC["missed_stunted_pct_mean"], 0.34, color=PAL[0], label="Stunting missed")
for i, s in enumerate(SIGMAS):
    ax[1].text(i, max(TC.loc[s, "false_stunted_pct_mean"], TC.loc[s, "missed_stunted_pct_mean"]) + 0.15,
               f"prev {TC.loc[s, 'prev_obs_pct_mean']:.2f}%", ha="center", fontsize=7)
ax[1].set_xticks(x); ax[1].set_xticklabels([f"{s} cm" for s in SIGMAS]); ax[1].set_xlabel("Random measurement error SD")
ax[1].set_ylabel("% of children"); ax[1].set_title(f"(b) Error direction (true prev {prev0:.2f}%)"); ax[1].legend(fontsize=8)
fig.savefig(os.path.join(FIG_DIR, "F09_montecarlo_misclassification.png")); plt.close(fig)

# F10: burden-capture trade-off curves
fig, ax = plt.subplots(1, 3, figsize=(10.5, 3.4), sharey=True)
for a, sig in zip(ax, SIGMAS):
    c = CV[CV.sigma_cm == sig]
    fb = c[c.policy == "Fixed HAZ band"]; pp = c[c.policy == "Proposed probability band"]
    a.plot(fb.burden, fb.capture, "-", color=PAL[1], lw=1.8, label="Fixed HAZ band (vary width)")
    a.plot(pp.burden, pp.capture, "-o", color=PAL[0], lw=1.8, ms=3.5, label="Proposed per-child band (vary alpha)")
    d1 = fb[np.isclose(fb.param, 0.10)].iloc[0]; d2 = pp[np.isclose(pp.param, ALPHA_MAIN)].iloc[0]
    a.plot(d1.burden, d1.capture, "s", color=PAL[1], ms=8, mec="k"); a.annotate("draft +/-0.10 SD", (d1.burden, d1.capture), xytext=(4, -12), textcoords="offset points", fontsize=7)
    a.plot(d2.burden, d2.capture, "D", color=PAL[0], ms=8, mec="k"); a.annotate(f"alpha={ALPHA_MAIN}", (d2.burden, d2.capture), xytext=(4, -12), textcoords="offset points", fontsize=7)
    a.set_xlim(0, 30); a.set_title(f"Error SD = {sig} cm"); a.set_xlabel("Children re-measured (%)")
ax[0].set_ylabel("Label errors captured (%)"); ax[0].legend(fontsize=7, loc="lower right")
fig.savefig(os.path.join(FIG_DIR, "F10_policy_tradeoff.png")); plt.close(fig)
print("done")


In [ ]:
!cd "{PROJ}/scripts" && SKI_PROJECT_DIR="{PROJ}" python 03_measurement_uncertainty.py

## Stage 4 - derivation and evaluation of the five categories
`04_decision_framework.py`

In [ ]:
%%writefile {PROJ}/scripts/04_decision_framework.py
"""
04_decision_framework.py
Stage 4 - Derivation and evaluation of the five screening categories.

Axis A (measurement certainty, per child):
    p_i = P(true HAZ < -2 | observed HAZ, sigma_cm)   = Phi((-2 - HAZ_obs) / (sigma_cm * z_per_cm_i))
    A1 confirmed stunted      p_i >= 1 - alpha
    A2 uncertain              alpha < p_i < 1 - alpha
    A3 confirmed not stunted  p_i <= alpha
Axis B (calibrated, leakage-free ML risk; threshold locked on development OOF):
    B1 high risk  P_cal >= t*      B0 low risk  P_cal < t*
Cells: 3 x 2 = 6. For A1 the action does not depend on B (the child is already stunted
and must be managed), so A1xB1 and A1xB0 merge -> 5 mutually exclusive, exhaustive categories.

Also applies the DRAFT matrix (fixed +/-0.10 SD, P_cal 0.30/0.60) to the same children
to show its coverage gaps.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from scipy.stats import norm
from common import *
from plot_style import apply_style, COLORBLIND_SAFE_PALETTE as PAL

apply_style(10)
rng = np.random.default_rng(SEED)
P = pd.read_pickle(os.path.join(DATA_DIR, "predictions.pkl"))
CFG = json.load(open(os.path.join(RES_DIR, "config_locked.json")))
T_RISK = CFG["risk_threshold"]

CAT = {1: "C1 Confirmed stunting", 2: "C2 Uncertain, high risk", 3: "C3 Uncertain, low risk",
       4: "C4 Not stunted, high risk", 5: "C5 Not stunted, low risk"}
ACTION = {1: "Refer to health centre for clinical assessment & stunting management; enrol in PMT/nutrition programme",
          2: "Priority re-measurement within 14 days by a second trained measurer (standard board); provisional counselling",
          3: "Re-measure at the next monthly posyandu session; routine counselling",
          4: "Not stunted: intensified prevention - monthly growth monitoring, counselling on modifiable risks (feeding, WASH, ANC)",
          5: "Routine growth monitoring and health education"}
COLORS = {1: "#B2182B", 2: "#EF8A62", 3: "#F4C28F", 4: "#67A9CF", 5: "#2166AC"}


def categorise(o, sigma=SIGMA_CM_MAIN, alpha=ALPHA_MAIN, t=T_RISK, haz=None):
    haz = o.haz.values if haz is None else haz
    p = norm.cdf((STUNT_CUT - haz) / (sigma * o.z_per_cm.values))
    hi = o.p_cal.values >= t
    c = np.where(p >= 1 - alpha, 1, np.where(p <= alpha, np.where(hi, 4, 5), np.where(hi, 2, 3)))
    return c, p


def draft_categorise(o):
    """Draft manuscript matrix. Methods version (HAZ-only outside zone) and
    Results-table version (also requires P_cal ranges outside zone)."""
    z, p = o.haz.values, o.p_cal.values
    inb = (z >= -2.10) & (z <= -1.90)
    meth = np.select([z < -2.10, z > -1.90, inb & (p >= .6), inb & (p >= .3), inb], [1, 5, 2, 3, 4], 0)
    res = np.select([(z < -2.10) & (p >= .6), inb & (p >= .6), inb & (p >= .3) & (p < .6), inb & (p < .3),
                     (z > -1.90) & (p < .3)], [1, 2, 3, 4, 5], 0)   # 0 = no category assigned
    return meth, res


rows, prof_rows, stab_rows = [], [], []
for part in ("test", "hold"):
    o = P[part].copy(); w = o[W].values
    c, p = categorise(o); o["cat"] = c; o["p_stunt_true"] = p
    o["draft_meth"], o["draft_res"] = draft_categorise(o)
    P[part] = o
    # stability: simulate one independent re-measurement (sigma = 1 cm) and re-categorise, 50 reps
    same = np.zeros(len(o))
    for r in range(50):
        hz2 = o.haz.values + rng.normal(0, SIGMA_CM_MAIN, len(o)) * o.z_per_cm.values
        c2, _ = categorise(o, haz=hz2)
        same += (c2 == c)
    o["stab"] = same / 50
    for k in range(1, 6):
        m = c == k
        if m.sum() == 0:
            continue
        ww = w[m]
        prof_rows.append(dict(partition="Locked test" if part == "test" else "Spatial holdout", category=CAT[k],
                              n=int(m.sum()), weighted_pct=100 * ww.sum() / w.sum(),
                              observed_stunting_pct=100 * np.average(o.stunted.values[m], weights=ww),
                              mean_haz=np.average(o.haz.values[m], weights=ww),
                              pct_haz_below_minus1=100 * np.average(o.haz.values[m] < -1, weights=ww),
                              mean_p_cal=np.average(o.p_cal.values[m], weights=ww),
                              mean_label_error_prob=100 * np.average(np.minimum(o.p_stunt_true.values[m], 1 - o.p_stunt_true.values[m]), weights=ww),
                              category_stability_pct=100 * np.average(o.stab.values[m], weights=ww),
                              action=ACTION[k]))
PR = pd.DataFrame(prof_rows).round(3)
PR.to_csv(os.path.join(RES_DIR, "T_category_profile.csv"), index=False)
print(PR.drop(columns="action").to_string())

# draft coverage
dr = []
for part in ("test", "hold"):
    o = P[part]; w = o[W].values
    for ver in ("draft_meth", "draft_res"):
        v = o[ver].values
        dr.append(dict(partition=part, version="Draft (Methods text)" if ver == "draft_meth" else "Draft (Results Table)",
                       unassigned_pct=100 * w[v == 0].sum() / w.sum(),
                       **{f"cat{k}_pct": 100 * w[v == k].sum() / w.sum() for k in range(1, 6)}))
    c = o.cat.values
    dr.append(dict(partition=part, version="Proposed", unassigned_pct=0.0,
                   **{f"cat{k}_pct": 100 * w[c == k].sum() / w.sum() for k in range(1, 6)}))
DR = pd.DataFrame(dr).round(3); DR.to_csv(os.path.join(RES_DIR, "T_draft_vs_proposed_coverage.csv"), index=False)
print(DR.to_string())

# sensitivity of category distribution to sigma and alpha (locked test)
sens = []
o = P["test"]; w = o[W].values
for sig in SIGMA_CM_SCENARIOS:
    for a in ALPHA_GRID:
        c, _ = categorise(o, sigma=sig, alpha=a)
        sens.append(dict(sigma_cm=sig, alpha=a, **{CAT[k]: 100 * w[c == k].sum() / w.sum() for k in range(1, 6)},
                         remeasure_pct=100 * w[(c == 2) | (c == 3)].sum() / w.sum()))
SE = pd.DataFrame(sens).round(2); SE.to_csv(os.path.join(RES_DIR, "T_category_sensitivity.csv"), index=False)
print(SE.to_string())

# cross-tab draft (methods) vs proposed
ct = pd.crosstab(P["test"].draft_meth.map({0: "none", 1: "D1 Stable Stunted", 2: "D2 Likely Stunted", 3: "D3 Uncertain",
                                            4: "D4 Likely Non-Stunted", 5: "D5 Stable Non-Stunted"}),
                 P["test"].cat.map(CAT), values=P["test"][W], aggfunc="sum", normalize="all") * 100
ct.round(2).to_csv(os.path.join(RES_DIR, "T_crosstab_draft_vs_proposed.csv")); print(ct.round(2))

# ================================================================== FIGURES
o = P["test"]; w = o[W].values
# F12 decision space: HAZ vs P_cal coloured by category
fig, ax = plt.subplots(1, 2, figsize=(10, 3.9), gridspec_kw=dict(width_ratios=[1.35, 1]))
sub = o.sample(min(6000, len(o)), random_state=SEED)
for k in range(5, 0, -1):
    s = sub[sub.cat == k]
    ax[0].scatter(s.haz, s.p_cal, s=4, alpha=.55, color=COLORS[k], label=CAT[k], rasterized=True)
ax[0].axhline(T_RISK, color="k", ls="--", lw=1); ax[0].axvline(-2, color="k", lw=.8)
ax[0].text(1.9, T_RISK + .01, f"t* = {T_RISK:.3f}", ha="right", fontsize=7)
ax[0].set_xlim(-5, 2); ax[0].set_xlabel("Observed HAZ"); ax[0].set_ylabel("Calibrated risk P_cal (ML, no anthropometry)")
ax[0].set_title("(a) Decision space, locked test"); ax[0].legend(fontsize=7, markerscale=3, loc="upper right")
pr = PR[PR.partition == "Locked test"]
yy = np.arange(5)[::-1]
ax[1].barh(yy, pr.weighted_pct, color=[COLORS[k] for k in range(1, 6)], height=.6)
for y_, (_, r) in zip(yy, pr.iterrows()):
    ax[1].text(r.weighted_pct + .5, y_, f"{r.weighted_pct:.1f}%  (stunted {r.observed_stunting_pct:.0f}%, stable {r.category_stability_pct:.0f}%)", va="center", fontsize=7)
ax[1].set_yticks(yy); ax[1].set_yticklabels(pr.category, fontsize=8); ax[1].set_xlim(0, 80)
ax[1].set_xlabel("Weighted share of children (%)"); ax[1].set_title("(b) Category distribution")
fig.savefig(os.path.join(FIG_DIR, "F12_decision_space.png")); plt.close(fig)

# F11 derivation matrix 3x2 -> 5
fig, ax = plt.subplots(figsize=(8.6, 4.2)); ax.axis("off"); ax.set_xlim(-0.2, 10); ax.set_ylim(0, 6.2)
ax.text(5.9, 5.95, "Axis B: calibrated ML risk (pre-measurement, no anthropometry)", ha="center", fontsize=9, weight="bold")
ax.text(4.35, 5.45, f"High risk  (P_cal >= {T_RISK:.2f})", ha="center", fontsize=8.5)
ax.text(7.55, 5.45, f"Low risk  (P_cal < {T_RISK:.2f})", ha="center", fontsize=8.5)
ax.text(0.15, 2.6, "Axis A: measurement certainty\nP(true HAZ < -2 | observed, sigma)", fontsize=8.5, weight="bold", va="center", ha="center", rotation=90)
rowsA = [("A1 Confirmed stunted\np >= 1 - alpha", 4.1), ("A2 Uncertain\nalpha < p < 1 - alpha", 2.55), ("A3 Confirmed not stunted\np <= alpha", 1.0)]
for lbl, y_ in rowsA:
    ax.text(2.55, y_ + .55, lbl, ha="right", va="center", fontsize=8)
pct = pr.set_index("category").weighted_pct
def cell(x, y_, wdt, k, txt):
    ax.add_patch(FancyBboxPatch((x, y_), wdt, 1.1, boxstyle="round,pad=0.02", fc=COLORS[k], ec="white", lw=2, alpha=.9))
    ax.text(x + wdt / 2, y_ + .55, f"{CAT[k]}\n{txt}\n({pct[CAT[k]]:.1f}% of children)", ha="center", va="center",
            fontsize=7.6, color="white" if k in (1, 5) else "black", weight="bold")
cell(2.75, 4.1, 6.4, 1, "Action: refer & manage (risk tier does not change action)")
cell(2.75, 2.55, 3.15, 2, "Priority re-measurement <= 14 d")
cell(6.0, 2.55, 3.15, 3, "Re-measure at next posyandu")
cell(2.75, 1.0, 3.15, 4, "Intensified prevention")
cell(6.0, 1.0, 3.15, 5, "Routine monitoring")
ax.text(5.9, 0.35, f"3 x 2 = 6 cells; the two A1 cells share one action -> 5 mutually exclusive, exhaustive categories "
        f"(sigma = {SIGMA_CM_MAIN} cm, alpha = {ALPHA_MAIN})", ha="center", fontsize=7.5, style="italic")
fig.savefig(os.path.join(FIG_DIR, "F11_category_derivation.png")); plt.close(fig)

# F13 draft vs proposed coverage
fig, ax = plt.subplots(figsize=(8.5, 2.9))
d = DR[DR.partition == "test"].set_index("version")
left = np.zeros(len(d))
labels = ["unassigned"] + [f"cat{k}_pct" for k in range(1, 6)]
cols = ["#BBBBBB"] + [COLORS[k] for k in range(1, 6)]
names = ["No category (gap)", "Cat 1", "Cat 2", "Cat 3", "Cat 4", "Cat 5"]
for lab, col, nm in zip(["unassigned_pct"] + labels[1:], cols, names):
    ax.barh(d.index, d[lab], left=left, color=col, label=nm, height=.55, edgecolor="white")
    left += d[lab].values
for i, v in enumerate(d.unassigned_pct):
    if v > 1:
        ax.text(v / 2, i, f"{v:.1f}%", ha="center", va="center", fontsize=8, weight="bold")
ax.set_xlabel("Weighted share of children, locked test (%)"); ax.set_xlim(0, 100)
ax.legend(ncol=6, fontsize=7, loc="upper center", bbox_to_anchor=(.5, 1.25))
fig.savefig(os.path.join(FIG_DIR, "F13_draft_vs_proposed.png")); plt.close(fig)

# F14 sensitivity heat table: remeasure share by sigma x alpha
fig, ax = plt.subplots(figsize=(4.6, 2.8))
hm = SE.pivot(index="sigma_cm", columns="alpha", values="remeasure_pct")
im = ax.imshow(hm.values, cmap="Blues", aspect="auto")
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        ax.text(j, i, f"{hm.values[i, j]:.1f}%", ha="center", va="center", fontsize=9,
                color="white" if hm.values[i, j] > hm.values.max() * .6 else "black")
ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels(hm.columns); ax.set_yticks(range(hm.shape[0]))
ax.set_yticklabels([f"{s} cm" for s in hm.index]); ax.set_xlabel("alpha (tolerated label-error probability)")
ax.set_ylabel("Assumed error SD"); ax.set_title("Children routed to re-measurement (C2+C3)"); ax.grid(False)
fig.savefig(os.path.join(FIG_DIR, "F14_sensitivity_heatmap.png")); plt.close(fig)
print("done")


In [ ]:
!cd "{PROJ}/scripts" && SKI_PROJECT_DIR="{PROJ}" python 04_decision_framework.py

## Flowcharts
`05_flowcharts.py`

In [ ]:
%%writefile {PROJ}/scripts/05_flowcharts.py
"""
05_flowcharts.py - methodological flowchart (Fig. 1), sample-flow diagram (Fig. 2)
and operational screening workflow (Fig. 15), drawn with matplotlib so they
reproduce in Colab without extra software.
"""
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon
from common import *

plt.rcParams.update({"font.family": "DejaVu Sans", "savefig.dpi": 300, "savefig.bbox": "tight"})
C = dict(data="#DCE9F5", proc="#FFFFFF", key="#FDE3C8", dec="#FFF4C2", out="#D9EFD9", edge="#333333")


def box(ax, x, y, w, h, txt, fc, fs=7.4, bold=False):
    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.015,rounding_size=0.08",
                                fc=fc, ec=C["edge"], lw=1))
    ax.text(x, y, txt, ha="center", va="center", fontsize=fs, weight="bold" if bold else "normal", wrap=True)


def diamond(ax, x, y, w, h, txt, fs=7):
    ax.add_patch(Polygon([(x, y + h / 2), (x + w / 2, y), (x, y - h / 2), (x - w / 2, y)], closed=True,
                         fc=C["dec"], ec=C["edge"], lw=1))
    ax.text(x, y, txt, ha="center", va="center", fontsize=fs)


def arrow(ax, x1, y1, x2, y2, txt=None, off=(0.08, 0)):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=C["edge"], lw=1))
    if txt:
        ax.text((x1 + x2) / 2 + off[0], (y1 + y2) / 2 + off[1], txt, fontsize=6.5, style="italic")


# ------------------------------------------------------------------ Fig 1 methodology
fig, ax = plt.subplots(figsize=(8.4, 9.8)); ax.axis("off"); ax.set_xlim(-0.1, 10.1); ax.set_ylim(2.9, 22)
ax.text(5, 21.6, "Research methodology flowchart", ha="center", fontsize=10, weight="bold")
box(ax, 5, 20.8, 7.2, .9, "SKI 2023 under-five microdata (86,364 children, 171 variables)\n+ codebook + WHO LMS length/height-for-age table", C["data"], bold=True)
arrow(ax, 5, 20.35, 5, 19.85)
box(ax, 5, 19.35, 7.2, 1.0, "STAGE 1 - Ingestion & audit: date-based age (days), WHO position correction (+/-0.7 cm),\nHAZ by LMS, |HAZ|>6 flags, sentinel codes (88/888/8888), duplicates, design variables", C["proc"])
arrow(ax, 5, 18.85, 5, 18.35)
diamond(ax, 5, 17.7, 4.6, 1.3, "Variable audit:\ndoes it form, co-measure,\nor follow the outcome?")
arrow(ax, 7.3, 17.7, 7.95, 17.7, "yes", off=(-0.35, 0.12))
box(ax, 8.95, 17.7, 1.95, 1.45, "Excluded\n(T1 outcome,\nT2 concurrent\nanthropometry,\nT3 proxies)", "#F6D5D5", fs=6.5)
arrow(ax, 5, 17.05, 5, 16.55, "no")
box(ax, 5, 16.1, 7.2, .8, "Locked predictor set: 121 pre-measurement features in 8 domains", C["key"], bold=True)
arrow(ax, 5, 15.7, 5, 15.2)
box(ax, 5, 14.7, 8.6, 1.0, "Design-aware split: spatial holdout (West Sumatra, Bali, Maluku) | locked test (20% of PSUs)\n| development set with 5-fold PSU-grouped cross-validation", C["proc"])
# two branches
arrow(ax, 3.2, 14.2, 2.6, 13.75); arrow(ax, 6.8, 14.2, 7.4, 13.75)
ax.text(2.6, 13.45, "AXIS B - risk (ML)", ha="center", fontsize=8, weight="bold", color="#8A3B00")
ax.text(7.4, 13.45, "AXIS A - measurement certainty", ha="center", fontsize=8, weight="bold", color="#0B4F8A")
box(ax, 2.6, 12.6, 4.4, 1.2, "STAGE 2a - Candidate models:\nLR, RF, HGB (weighted/unweighted)\n+ leaky & random-CV comparators", C["proc"])
box(ax, 7.4, 12.6, 4.4, 1.2, "STAGE 3a - Error propagation:\nSD units per 1 cm by age & sex\n(WHO LMS: dz/dh = 1/(M*S))", C["proc"])
arrow(ax, 2.6, 12.0, 2.6, 11.5); arrow(ax, 7.4, 12.0, 7.4, 11.5)
box(ax, 2.6, 10.9, 4.4, 1.2, "STAGE 2b - Platt / isotonic calibration\nfitted on out-of-fold predictions only;\nthreshold t* fixed on OOF (Youden J)", C["proc"])
box(ax, 7.4, 10.9, 4.4, 1.2, "STAGE 3b - Stress test in cm (+/-0.5, +/-1.0,\n+/-0.7 position) & Monte Carlo errors\n(sigma = 0.5 / 1.0 / 1.5 cm, 100 reps)", C["proc"])
arrow(ax, 2.6, 10.3, 2.6, 9.8); arrow(ax, 7.4, 10.3, 7.4, 9.8)
box(ax, 2.6, 9.2, 4.4, 1.2, "Evaluation: weighted AUROC, AUPRC, Brier,\nECE, calibration slope/intercept, PSU\nbootstrap CI, DCA, subgroups, importance", C["out"])
box(ax, 7.4, 9.2, 4.4, 1.2, "Re-measurement policy comparison:\nP0 none | P1 draft +/-0.10 SD | P2 budget-\nmatched fixed band | P3 per-child band", C["out"])
arrow(ax, 2.6, 8.6, 4.2, 7.75); arrow(ax, 7.4, 8.6, 5.8, 7.75)
box(ax, 5, 7.2, 9.6, 1.1, "STAGE 4 - Decision framework: Axis A (3 levels, alpha) x Axis B (2 levels, t*) = 6 cells\n-> merge the two 'confirmed stunted' cells -> 5 mutually exclusive, exhaustive categories", C["key"], bold=True)
arrow(ax, 5, 6.65, 5, 6.15)
box(ax, 5, 5.6, 7.2, 1.0, "Category validation on locked test & spatial holdout: observed stunting, mean HAZ,\nlabel-error probability, re-measurement stability, sensitivity to sigma & alpha,\ncomparison with the draft matrix", C["proc"])
arrow(ax, 5, 5.1, 5, 4.6)
box(ax, 5, 4.05, 9.6, 1.0, "Output: 5 screening categories with graded actions\n(refer & manage | priority re-measure | routine re-measure | intensified prevention | routine monitoring)", C["out"], bold=True)
ax.text(5, 3.2, "Blue = data; white = processing; yellow = decision; orange = locked/derived artefact; green = evaluation/output",
        ha="center", fontsize=6.5, color="#555")
fig.savefig(os.path.join(FIG_DIR, "F01_methodology_flowchart.png")); plt.close(fig)

# ------------------------------------------------------------------ Fig 2 sample flow
fl = pd.read_csv(os.path.join(RES_DIR, "T_flow_exclusion.csv"))
sp = pd.read_csv(os.path.join(RES_DIR, "T_split_summary.csv")).set_index("split")
n = dict(zip(fl.step, fl.n))
fig, ax = plt.subplots(figsize=(8, 4.9)); ax.axis("off"); ax.set_xlim(-0.1, 10.1); ax.set_ylim(3.9, 11.7)
box(ax, 4, 11.2, 5, .8, f"SKI 2023 under-five records\nn = {n['Raw SKI 2023 under-five records']:,}", C["data"], fs=8, bold=True)
arrow(ax, 4, 10.8, 4, 9.5)
box(ax, 8, 10.15, 3.4, .9, f"Excluded: length/height not measured\nn = {n['Raw SKI 2023 under-five records'] - n['Length/height measured']:,}", "#F6D5D5", fs=7)
arrow(ax, 4, 10.15, 6.3, 10.15)
box(ax, 4, 9.1, 5, .8, f"Length/height measured, age 0-59 months\nn = {n['Length/height measured']:,}", C["proc"], fs=8)
arrow(ax, 4, 8.7, 4, 7.4)
box(ax, 8, 8.05, 3.4, .9, f"Excluded: implausible HAZ (|HAZ| > 6)\nn = {n['Excluded biologically implausible HAZ (|HAZ|>6, WHO flag)']:,}", "#F6D5D5", fs=7)
arrow(ax, 4, 8.05, 6.3, 8.05)
box(ax, 4, 7.0, 6.2, .8, f"Final analytic sample (complete design variables)\nn = {n['Final analytic sample (unique child ID)']:,}", C["key"], fs=8, bold=True)
for x, key, lbl in [(1.6, "dev", "Development set"), (4.9, "test", "Locked test set"), (8.2, "holdout", "Spatial holdout")]:
    arrow(ax, 4, 6.6, x, 5.5)
    r = sp.loc[key]
    extra = "\n(W. Sumatra, Bali, Maluku)" if key == "holdout" else ("\n(20% of PSUs)" if key == "test" else "\n(5-fold PSU-grouped CV)")
    box(ax, x, 4.8, 3.0, 1.3, f"{lbl}{extra}\nn = {int(r.n):,}; PSU = {int(r.n_psu):,}\nweighted stunting {100 * r.prev_w:.1f}%", C["out"], fs=7)
fig.savefig(os.path.join(FIG_DIR, "F02_sample_flow.png")); plt.close(fig)

# ------------------------------------------------------------------ Fig 15 operational workflow
fig, ax = plt.subplots(figsize=(9, 4.6)); ax.axis("off"); ax.set_xlim(0, 12); ax.set_ylim(0, 6)
box(ax, 1.2, 4.6, 2.1, 1.1, "Household & maternal\ninformation\n(no anthropometry)", C["data"], fs=7)
arrow(ax, 2.25, 4.6, 2.95, 4.6)
box(ax, 4.05, 4.6, 2.1, 1.1, "Calibrated ML risk\nP_cal -> high / low\n(Axis B)", C["key"], fs=7)
arrow(ax, 5.1, 4.6, 5.8, 4.6)
ax.text(5.45, 5.35, "prioritise\nwho is measured first", ha="center", fontsize=6.3, style="italic")
box(ax, 6.9, 4.6, 2.1, 1.1, "Standardised length/\nheight measurement\n+ age in days", C["proc"], fs=7)
arrow(ax, 7.95, 4.6, 8.55, 4.6)
box(ax, 9.9, 4.6, 2.6, 1.1, "p = P(true HAZ < -2 |\nobserved, sigma_cm)\n(Axis A)", C["key"], fs=7)
arrow(ax, 9.9, 4.05, 9.9, 3.45)
diamond(ax, 9.9, 2.85, 2.6, 1.1, "p >= 1-alpha ?\nalpha < p < 1-alpha ?\np <= alpha ?", fs=6.3)
cols = {1: "#B2182B", 2: "#EF8A62", 3: "#F4C28F", 4: "#67A9CF", 5: "#2166AC"}
lab = {1: "C1 Refer &\nmanage", 2: "C2 Priority\nre-measure", 3: "C3 Routine\nre-measure", 4: "C4 Intensified\nprevention", 5: "C5 Routine\nmonitoring"}
for i, k in enumerate(range(1, 6)):
    x = 1.1 + i * 1.75
    ax.add_patch(FancyBboxPatch((x - .75, .35), 1.5, .95, boxstyle="round,pad=0.02", fc=cols[k], ec="white"))
    ax.text(x, .82, lab[k], ha="center", va="center", fontsize=6.8, color="white" if k in (1, 5) else "black", weight="bold")
    ax.annotate("", xy=(x, 1.32), xytext=(8.6, 2.85), arrowprops=dict(arrowstyle="-|>", color="#777", lw=.7))
ax.annotate("", xy=(6.9, 4.05), xytext=(3.5, 1.32), arrowprops=dict(arrowstyle="-|>", color="#B2182B", lw=1, ls="--", connectionstyle="arc3,rad=-0.3"))
ax.annotate("", xy=(6.9, 4.05), xytext=(5.0, 1.32), arrowprops=dict(arrowstyle="-|>", color="#B2182B", lw=1, ls="--", connectionstyle="arc3,rad=-0.2"))
ax.text(4.9, 2.9, "second measurement;\naverage and re-classify", fontsize=6.5, color="#B2182B", style="italic")
fig.savefig(os.path.join(FIG_DIR, "F15_operational_workflow.png")); plt.close(fig)
print("flowcharts done")


In [ ]:
!cd "{PROJ}/scripts" && SKI_PROJECT_DIR="{PROJ}" python 05_flowcharts.py

## Show key tables and figures

In [ ]:
import pandas as pd, glob
from IPython.display import Image, display
for t in ['T_model_comparison_test_CI','T_policy_head_to_head','T_category_profile','T_draft_vs_proposed_coverage']:
    print(t); display(pd.read_csv(f'{PROJ}/results/{t}.csv'))
for f in sorted(glob.glob(PROJ + '/figures/*.png')):
    print(f); display(Image(f, width=800))